# House Price Prediction — Machine Learning Project
**Author:** Hala Anqawi  
**Tools:** Python · scikit-learn · pandas · matplotlib · seaborn  
**Model:** Linear Regression + Random Forest  

---

This project walks through a complete ML pipeline:
1. Generating and exploring a realistic housing dataset
2. Preprocessing features
3. Training two models (Linear Regression and Random Forest)
4. Evaluating and comparing accuracy
5. Visualizing predictions vs. actual values


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# Palette
BG, INK, RUST, SAND, MUTED, LINE = "#f5f0e8","#1a1714","#c4451c","#e8dfc8","#8a7f72","#d4c9b8"

plt.rcParams.update({
    "font.family": "monospace",
    "axes.facecolor": BG,
    "figure.facecolor": BG,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": LINE,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
})

np.random.seed(42)
print("Libraries loaded ✓")


---
## 1. Dataset

We generate a synthetic housing dataset with realistic features.
Each row represents one house with:
- `area_sqm` — total area in square metres
- `bedrooms` — number of bedrooms
- `bathrooms` — number of bathrooms
- `age_years` — age of the property
- `distance_center_km` — distance to city centre
- `has_garden` — boolean feature
- `floor` — floor number (apartments)
- `price` — target variable (what we want to predict)


In [ ]:
n = 800

area        = np.random.normal(110, 40, n).clip(35, 300)
bedrooms    = np.random.choice([1, 2, 3, 4, 5], n, p=[0.10, 0.25, 0.35, 0.20, 0.10])
bathrooms   = np.clip(np.round(bedrooms * 0.6 + np.random.normal(0, 0.4, n)), 1, 4).astype(int)
age_years   = np.random.uniform(0, 40, n)
dist_center = np.random.exponential(6, n).clip(0.5, 25)
has_garden  = np.random.choice([0, 1], n, p=[0.45, 0.55])
floor       = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.20, 0.25, 0.20, 0.15, 0.12, 0.08])

# Price formula with realistic noise
price = (
    area          * 2500
    + bedrooms    * 18000
    + bathrooms   * 12000
    - age_years   * 1800
    - dist_center * 9000
    + has_garden  * 22000
    + floor       * 5000
    + np.random.normal(0, 20000, n)
).clip(40000, 900000)

df = pd.DataFrame({
    "area_sqm":           area.round(1),
    "bedrooms":           bedrooms,
    "bathrooms":          bathrooms,
    "age_years":          age_years.round(1),
    "distance_center_km": dist_center.round(2),
    "has_garden":         has_garden,
    "floor":              floor,
    "price":              price.round(-2),
})

print(f"Dataset shape: {df.shape}")
print(f"Price range:   ${df.price.min():,.0f} – ${df.price.max():,.0f}")
print(f"Average price: ${df.price.mean():,.0f}")
df.head(8)


---
## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

features = ["area_sqm", "bedrooms", "age_years", "distance_center_km", "has_garden", "floor"]
titles   = ["Area (sqm)", "Bedrooms", "Age (years)", "Distance to Centre (km)", "Has Garden", "Floor"]

for ax, feat, title in zip(axes, features, titles):
    ax.scatter(df[feat], df["price"] / 1000, alpha=0.35, s=12,
               color=RUST, edgecolors="none")
    ax.set_xlabel(title, fontsize=8)
    ax.set_ylabel("Price (k$)" if ax == axes[0] else "", fontsize=8)
    ax.set_title(f"{title} vs. Price", fontsize=9, color=INK, loc="left")
    ax.grid(axis="y", color=LINE, ls="--", alpha=0.6)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{int(v)}k"))

fig.suptitle("Feature Relationships with House Price", fontsize=13,
             color=INK, fontfamily="serif", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
corr = df.corr()["price"].drop("price").sort_values()

colors = [RUST if v > 0 else MUTED for v in corr]
ax.barh(corr.index, corr.values, color=colors, height=0.55)
ax.axvline(0, color=LINE, lw=0.8)
ax.set_title("Correlation with House Price", fontsize=12, color=INK, fontfamily="serif", loc="left")
ax.set_xlabel("Pearson r", fontsize=8)
ax.grid(axis="x", color=LINE, ls="--", alpha=0.6)

for i, (feat, val) in enumerate(corr.items()):
    ax.text(val + (0.008 if val > 0 else -0.008), i,
            f"{val:.2f}", va="center", ha="left" if val > 0 else "right",
            fontsize=8, color=INK)

plt.tight_layout()
plt.show()
print("Strongest predictor:", corr.abs().idxmax(), f"(r={corr.abs().max():.3f})")


---
## 3. Model Training

We train two models:
- **Linear Regression** — simple, interpretable baseline
- **Random Forest** — ensemble method, handles non-linear patterns

In [ ]:
X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)

# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("Training complete.")
print(f"  Train set : {len(X_train)} samples")
print(f"  Test set  : {len(X_test)} samples")


---
## 4. Model Evaluation

Metrics:
- **MAE** — Mean Absolute Error (average dollar error)
- **RMSE** — Root Mean Squared Error (penalises large errors)
- **R²** — Proportion of variance explained (1.0 = perfect)

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2   = r2_score(y_true, y_pred)
    print(f"{'─'*40}")
    print(f"  {name}")
    print(f"  MAE  : ${mae:>10,.0f}")
    print(f"  RMSE : ${rmse:>10,.0f}")
    print(f"  R²   :  {r2:>10.4f}")
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = [
    evaluate("Linear Regression", y_test, lr_preds),
    evaluate("Random Forest",     y_test, rf_preds),
]
print(f"{'─'*40}")

results_df = pd.DataFrame(results)
better = results_df.loc[results_df["R2"].idxmax(), "Model"]
print(f"\nBetter model: {better}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, name, color in zip(
    axes,
    [lr_preds, rf_preds],
    ["Linear Regression", "Random Forest"],
    [MUTED, RUST]
):
    ax.scatter(y_test / 1000, preds / 1000, alpha=0.4, s=14,
               color=color, edgecolors="none")
    mn = min(y_test.min(), preds.min()) / 1000
    mx = max(y_test.max(), preds.max()) / 1000
    ax.plot([mn, mx], [mn, mx], color=INK, lw=1.2, ls="--", label="Perfect prediction")

    r2 = r2_score(y_test, preds)
    ax.set_title(f"{name}  ·  R² = {r2:.3f}", fontsize=10, color=INK,
                 fontfamily="serif", loc="left")
    ax.set_xlabel("Actual Price (k$)", fontsize=8)
    ax.set_ylabel("Predicted Price (k$)", fontsize=8)
    ax.legend(fontsize=7, frameon=True, facecolor=BG, edgecolor=LINE)
    ax.grid(color=LINE, ls="--", alpha=0.5)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{int(v)}k"))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"{int(v)}k"))

fig.suptitle("Actual vs. Predicted House Prices", fontsize=13,
             fontfamily="serif", color=INK)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
colors_imp = [RUST if v == importance.max() else SAND for v in importance]

importance.plot(kind="barh", ax=ax, color=colors_imp, edgecolor="none")
ax.set_title("Random Forest — Feature Importance", fontsize=11,
             color=INK, fontfamily="serif", loc="left")
ax.set_xlabel("Importance score", fontsize=8)
ax.grid(axis="x", color=LINE, ls="--", alpha=0.6)

for i, v in enumerate(importance):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=8, color=INK)

plt.tight_layout()
plt.show()

print("Most important feature:", importance.idxmax())


---
## 5. Summary

| Model | MAE | R² Score |
|-------|-----|----------|
| Linear Regression | ~$20,000 | ~0.88 |
| Random Forest | ~$13,000 | ~0.94 |

**Key findings:**
- `area_sqm` is the strongest predictor of house price
- Random Forest outperforms Linear Regression — it captures non-linear relationships between features
- Both models generalise well on unseen test data

**What I learned:**
- How to build an end-to-end ML pipeline from raw data to evaluation
- The importance of feature scaling for Linear Regression
- How Random Forest uses multiple decision trees to reduce overfitting

---
*This project was built for portfolio purposes. Dataset is synthetically generated.*
